In [41]:
import csv
import pandas as pd
input_path = "/home/oso/code/spar_deception/data/datasets/synthetic/ai_safety_case_studies.csv"
output_path = "/home/oso/code/spar_deception/data/datasets/synthetic/ai_safety_case_studies_fixed.csv"

# 1. Define your original 14 headers
headers = [
    "entry_id","theme","game_type","source_type","system","user","content"

]

dataset = []

# --- STEP 1: READ AND REPAIR ---
with open(input_path, mode='r', encoding='utf-8') as csvfile:
    # 'restkey' catches all the data broken by commas into one list
    reader = csv.DictReader(csvfile, restkey='overflow')

    for row in reader:
        if 'overflow' in row and row['overflow']:
            # Reconstruct the content by joining the pieces back with commas
            row['content'] = row['content'] + "," + ",".join(row['overflow'])
            del row['overflow']
        
        dataset.append(row)

# --- STEP 2: WRITE PERMANENT FIX ---
with open(output_path, mode='w', newline='', encoding='utf-8') as csvfile:
    # QUOTE_MINIMAL is the key: it wraps 'content' in "" so it won't break again
    writer = csv.DictWriter(csvfile, fieldnames=headers, quoting=csv.QUOTE_MINIMAL)
    
    writer.writeheader()
    writer.writerows(dataset)

print(f"Repaired {len(dataset)} rows and saved to: {output_path}")

Repaired 10057 rows and saved to: /home/oso/code/spar_deception/data/datasets/synthetic/ai_safety_case_studies_fixed.csv


In [2]:
import json
import csv

input_file = "/home/oso/code/spar_deception/data/datasets/synthetic/safety_reports_model_named.jsonl" # Replace with your filename
output_file = "/home/oso/code/spar_deception/data/datasets/synthetic/safety_reports_model_named.csv"

# Define the headers for your CSV
headers = ["role_system", "user_input", "content"]

with open(input_file, 'r', encoding='utf-8') as f_in, \
     open(output_file, 'w', newline='', encoding='utf-8') as f_out:
    
    # Use QUOTE_MINIMAL to protect the story text from commas/newlines
    writer = csv.DictWriter(f_out, fieldnames=headers, quoting=csv.QUOTE_MINIMAL)
    writer.writeheader()
    
    for line in f_in:
        if not line.strip():
            continue
            
        data = json.loads(line)
        messages = data.get("messages", [])
        
        # Extracting based on the specific indices in your JSON structure
        # Index 0: System, Index 1: User, Index 2: Assistant
        row = {
            "role_system": messages[0]["content"] if len(messages) > 0 else "",
            "user_input": messages[1]["content"] if len(messages) > 1 else "",
            "content": messages[2]["content"] if len(messages) > 2 else ""
        }
        
        writer.writerow(row)

print(f"Transformation complete. Saved to {output_file}")

Transformation complete. Saved to /home/oso/code/spar_deception/data/datasets/synthetic/safety_reports_model_named.csv
